# Practice Lab: Prompt Optimization (Lesson 13.2)

Module 13 . 4 exercises . Audit + A/B test + context budget + CI pipeline


# Lesson 13.2: Prompt Optimization

4 exercises: 1E + 2M + 1C

Module 13: LLM Optimization & Self-Hosting


In [ ]:
import re, random, json
random.seed(42)


---
## Exercise 1 (Easy): Prompt Audit


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import re

class PA:
    BLOAT=[(r"please\s+(kindly\s+)?","Politeness"),(r"make sure (that |to )?","Redundant make sure"),
        (r"it is important (that |to )?","Filler important"),(r"I want you to","Unnecessary framing"),
        (r"you should always","Redundant should"),(r"(note|remember|keep in mind) that","Filler note"),
        (r"in order to","Wordy"),(r"as an AI (language |)model","Model identity"),
        (r"(here is|the following is) (a |an )?(list|example)","Narration")]
    def tokens(self,t): return len(t)//4+1
    def analyze(self,name,p,price=0.10,daily=10000):
        tok=self.tokens(p); bl=[]
        for pat,desc in self.BLOAT:
            m=re.findall(pat,p,re.IGNORECASE)
            if m: bl.append({"p":desc,"c":len(m)})
        cc=tok*price/1e6; cm=cc*daily*30
        return {"name":name,"tok":tok,"bl":len(bl),"bld":bl,"cm":round(cm,4),"ci":round(cm*84,2)}

a=PA()
prompts={"advisor":"""You are an AI language model assistant for Netsetos, an edtech company.\nI want you to please kindly help students with their course selection. It is important that you always recommend courses based on the student's goals.\nMake sure that you always provide accurate pricing. Note that our courses include:\n- GenAI Complete: Rs29,999\n- ML Engineering: Rs39,999\n- Python: Rs9,999\nYou should always be polite. Please make sure to ask clarifying questions. Remember that you represent Netsetos.\nHere is an example of a good response:\nStudent: "I want to learn AI"\nAssistant: "GenAI Complete at Rs29,999 covers fundamentals to production."\nIn order to provide the best experience, please keep responses concise.""",
    "code_review":"You are an AI language model that reviews Python code. I want you to please analyze for bugs. Make sure that you explain each issue. It is important that you provide corrections. Note that you should follow PEP 8. You should always suggest improvements.",
    "support":"Help users with account issues. Be polite. Escalate billing to humans.",
    "summarizer":"Summarize in under 3 sentences. Include key facts. Neutral tone.",
    "rag":"You are an AI language model assistant. I want you to please kindly answer from context. Make sure that you cite sources. It is important that you say I don't know if unsure. Note that answers should be concise. In order to maintain accuracy, ground in context."}

results=[a.analyze(n,p) for n,p in prompts.items()]
results.sort(key=lambda x:-x["bl"])
print("Prompt Audit:")
print(f"  {'Name':<14} {'Tok':>5} {'Bloat':>6} {'$/mo':>8} {'Rs/mo':>10}")
for r in results: print(f"  {r['name']:<14} {r['tok']:>5} {r['bl']:>6} ${r['cm']:>7} Rs{r['ci']:>8}")
print(f"\n  Worst: {results[0]['name']} ({results[0]['bl']} patterns)")
for b in results[0]["bld"]: print(f"    - {b['p']} (x{b['c']})")
total=sum(r["cm"] for r in results)
print(f"\n  Total: ${total:.4f}/mo Flash | ${total*12.5:.2f}/mo Pro")


</details>


---
## Exercise 2 (Medium): A/B Compression


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import re, random; random.seed(42)

class PC:
    def tokens(self,t): return len(t)//4+1
    def strip(self,p):
        for pat,rep in [(r"please\s+(kindly\s+)?",""),(r"I want you to\s+",""),(r"make sure (that |to )?",""),
            (r"it is important (that |to )?",""),(r"you should always\s+",""),(r"(note|remember|keep in mind) that\s+",""),
            (r"in order to","to"),(r"as an AI (language |)model\s*(assistant|)?",""),
            (r"here is (a |an )?(list|example)[^:]*:\s*","")]:
            p=re.sub(pat,rep,p,flags=re.IGNORECASE)
        return re.sub(r"\s+"," ",p).strip()
    def restructure(self):
        return "Role: Netsetos advisor\nCourses: GenAI Rs29,999 | ML Rs39,999 | Python Rs9,999\nRules: recommend by goals, include pricing, ask if unclear, be concise\nExample: Q: Learn AI -> A: GenAI Complete (Rs29,999) covers fundamentals to production."

ORIG="""You are an AI language model assistant for Netsetos. I want you to please kindly help students. It is important that you recommend based on goals. Make sure to include pricing. Note that courses: GenAI Rs29,999, ML Rs39,999, Python Rs9,999. You should always be polite. Please make sure to ask if unclear. Here is an example: Student: I want AI. Assistant: GenAI Rs29,999 covers it all. In order to be helpful, keep it concise."""
c=PC(); s1=c.strip(ORIG); s3=c.restructure(); ot=c.tokens(ORIG)
print("A/B Compression Test:")
for n,t in [("Original",ORIG),("Strip filler",s1),("Restructure",s3)]:
    tk=c.tokens(t); print(f"  {n:<20} {tk:>4} tok ({(ot-tk)/ot*100:>4.0f}% saved)")

print(f"\n  Quality (5 queries):")
os_l=[]; cs_l=[]
for q in ["Learn AI","Cheapest?","Budget Rs20K","Compare courses","Certificates?"]:
    o=random.uniform(7.5,9.5); cv=min(10,max(5,o+random.uniform(-0.3,0.2)))
    os_l.append(o); cs_l.append(cv)
    print(f"    {q:<18} orig={o:.1f} comp={cv:.1f} delta={cv-o:+.1f}")
ao=sum(os_l)/5; ac=sum(cs_l)/5; dp=abs(ac-ao)/ao*100
ft=c.tokens(s3); sv=(ot-ft)*0.10/1e6*10000*30
print(f"\n  Avg: {ao:.1f} vs {ac:.1f} delta={dp:.1f}%")
print(f"  Decision: {'SHIP' if dp<5 else 'REJECT'} | Saved: {ot-ft} tok | ${sv:.4f}/mo Flash")


</details>


---
## Exercise 3 (Medium): Dynamic Context Budget


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import random; random.seed(42)

class DCB:
    def __init__(self,sb=200,hb=800,qb=100): self.sb=sb; self.hb=hb; self.qb=qb; self.tb=sb+hb+qb
    def tok(self,t): return len(t)//4+1
    def manage(self,sys,turns,q):
        st=min(self.tok(sys),self.sb); qt=min(self.tok(q),self.qb); rem=self.hb
        kept=summ=drop=0
        for i,t in enumerate(reversed(turns)):
            tt=self.tok(t)
            if i<3:
                use=min(tt,rem); kept+=1; rem-=use
            elif i<10:
                use=min(max(10,tt//5),rem); summ+=1; rem-=max(0,use)
            else: drop+=1
        used=st+qt+(self.hb-rem); unb=st+qt+sum(self.tok(t) for t in turns)
        return {"used":used,"unb":unb,"util":round(used/self.tb*100),"k":kept,"s":summ,"d":drop,
            "sv":round((1-used/max(unb,1))*100)}

b=DCB(200,800,100)
sys="Role: Netsetos advisor. Courses: GenAI Rs29,999 | ML Rs39,999 | Python Rs9,999."
turns=[]
print("Dynamic Context Budget (15 turns):")
print(f"  {'Turn':>4} {'Unbound':>8} {'Budget':>8} {'Util':>5} {'K':>3} {'S':>3} {'D':>3} {'Save':>5}")
for i in range(15):
    turns.append(f"User: Question about turn {i+1} with some detail about courses and pricing")
    turns.append(f"Asst: Response to turn {i+1} with relevant course information and recommendations")
    r=b.manage(sys,turns,"Follow-up question")
    print(f"  {i+1:>4} {r['unb']:>8} {r['used']:>8} {r['util']:>4}% {r['k']:>3} {r['s']:>3} {r['d']:>3} {r['sv']:>4}%")
print(f"\n  After 15 turns: {r['unb']} unbound vs {r['used']} budgeted ({r['sv']}% saved)")


</details>


---
## Exercise 4 (Challenge): CI Auto-Optimize


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import re, random; random.seed(42)

class CIPO:
    def __init__(self,lim=80,qth=5.0): self.lim=lim; self.qth=qth
    def tok(self,t): return len(t)//4+1
    def strip(self,p):
        for pat,rep in [(r"please\s+(kindly\s+)?",""),(r"I want you to\s+",""),(r"make sure (that |to )?",""),
            (r"it is important (that |to )?",""),(r"you should always\s+",""),(r"(note|remember|keep in mind) that\s+",""),
            (r"in order to","to"),(r"as an AI (language |)model\s*(assistant|)?",""),
            (r"here is (a |an )?(list|example)[^:]*:\s*","")]:
            p=re.sub(pat,rep,p,flags=re.IGNORECASE)
        return re.sub(r"\s+"," ",p).strip()
    def qtest(self,o,c):
        scores=[(random.uniform(7.5,9.5),) for _ in range(5)]
        scores=[(s[0],min(10,max(5,s[0]+random.uniform(-0.3,0.2)))) for s in scores]
        ao=sum(s[0] for s in scores)/5; ac=sum(s[1] for s in scores)/5
        return round(abs(ac-ao)/ao*100,1), ac>=ao*0.95
    def run(self,prompts):
        res=[]
        for n,p in prompts.items():
            t=self.tok(p)
            if t<=self.lim: res.append({"n":n,"t":t,"s":"OK","a":"none","sv":0}); continue
            comp=self.strip(p); ct=self.tok(comp); sv=t-ct; sp=round(sv/t*100)
            qd,qp=self.qtest(p,comp)
            res.append({"n":n,"t":t,"ct":ct,"sv":sv,"sp":sp,"qd":qd,"qp":qp,"s":"FLAG","a":"compressed" if qp else "blocked"})
        return res

ci=CIPO(80,5.0)  # 80-token limit to trigger flagging
cb={"advisor":"You are an AI language model assistant for Netsetos. I want you to please kindly help students. It is important that you recommend based on goals. Make sure to include pricing. Note that courses: GenAI Rs29,999, ML Rs39,999, Python Rs9,999. You should always be polite. Please make sure to ask if unclear. Here is an example: Student: I want AI. Assistant: GenAI Rs29,999. In order to help, be concise.",
    "code_rev":"You are an AI language model. I want you to please review Python code. Make sure to explain bugs. It is important to provide corrections. Note that PEP 8 applies. You should always suggest improvements. Here is an example: bad code -> fix.",
    "rag_qa":"You are an AI language model. I want you to please answer from context only. Make sure not to hallucinate. It is important to cite sources. Note that say I don't know if unsure. In order to be accurate, ground in context.",
    "summarizer":"Summarize in 3 sentences. Include facts. Neutral tone.",
    "translator":"Translate to Hindi. Keep formatting.",
    "greeter":"Welcome user. Ask how to help. Be friendly."}

res=ci.run(cb); ts=0; ap=True
print("CI Pipeline (limit: 80 tokens):")
print(f"  {'Prompt':<12} {'Tok':>5} {'Status':>7} {'Action':>12} {'Saved':>7} {'QDelta':>7}")
for r in res:
    sv=f"{r.get('sp',0)}%" if r.get("sv") else "-"; qd=f"{r.get('qd',0):.1f}%" if r.get("qd") else "-"
    print(f"  {r['n']:<12} {r['t']:>5} {r['s']:>7} {r['a']:>12} {sv:>7} {qd:>7}")
    ts+=r.get("sv",0)
    if r["a"]=="blocked": ap=False
print(f"\n  Total saved: {ts} tok | Monthly: ${ts*0.10/1e6*10000*30:.4f}")
print(f"  Deploy: {'DEPLOY' if ap else 'BLOCKED'}")


</details>
